In [1]:
import pandas as pd
DATA_PATH = "amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"
df = pd. read_csv (DATA_PATH, sep="\t")
print ("Default settings:", df. shape)

Default settings: (144724, 15)


There are 144724 rows and 15 columns

In [2]:
df = pd. read_csv (DATA_PATH, sep="\t", quoting=3)
print("quoting=3:", df. shape)


quoting=3: (145431, 15)


Now it shows that there are 145431 rows
Pandas did not display a warning
This suggests that quotation marks in the original review text affected how the data was read.

This shows that just because a dataset loads successfully doesn't necessarily mean it was loaded correctly so it is important to check the number of rows and columns because errors in how the data is read could cause reviews to be missing without being immediately obvious.


In [3]:
peek = df[ "review_body"]. sample(5, random_state=2134) 
for text in peek:
    print (text,
"\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

There are random snippets of text such as "<br /><br />I" 
And there is unusual punctuation/formatting

Some of the reviews contain HTML tags like <escape><br /></escape> that aren't normally part of written text. These probably came from Amazon's website formatting, where the tags were used to create line breaks. So when the reviews were exported into a dataset, the formatting remained in the text.

In [4]:
df["star_rating"]. value_counts (normalize=True). sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

The star ratings are not evenly distributed, with a larger share of reviews receiving higher ratings

This matters because a model could achieve high accuracy by frequently predicting positive sentiment, even if it struggles to identify negative reviews

An uneven dataset could therefore make a model seem more accurate than it actually is.

In [5]:
df[ "product_title"]. value_counts().head (10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

The most-reviewed products aren't necessarily individual video games (some appear to be digital services, subscriptions, or other gaming-related products) 

This is interesting because the dataset is labeled as video game reviews, but it may include customers evaluating things other than gameplay

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())

df = df.dropna(subset=["review_body"])

df["review_body"] = df["review_body"].str.replace("<br />", " ")

print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

Before cleaning, 25,699 reviews contained <escape><br /></escape> line breaks. After cleaning, 0 reviews contained these line breaks, meaning they were successfully removed from the dataset.

Some reviews still contain strange formatting, like &#34; instead of quotation marks and extra backslashes, as well as spelling and grammar mistakes

I don't think these issues would significantly change whether the model interprets most reviews as positive or negative, since the overall sentiment is still pretty clear. However, unusual formatting or confusing wording could make it harder for the model to understand certain reviews accurately.

In [7]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)

complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)

print(len(sample), len(complaints))

500 20


Since the original dataset has an uneven distribution of star ratings, selecting 500 random reviews would likely include many more positive reviews than negative ones 

Taking 100 reviews from each rating makes the sample balanced, allowing us to evaluate how well the model performs across all levels of customer satisfaction instead of having the results dominated by the most common ratings.

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])

classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

The two keys are label and score. The label shows that the model classified the review as NEGATIVE, while the score of 0.9998 means the model is approximately 99.98% confident in its prediction. However, high confidence doesn't necessarily mean the prediction is correct.

The model predicted NEGATIVE sentiment with very high confidence

If the customer gave the review 1 or 2 stars, the model agrees with their rating, however, I would need to see the customer's actual star rating to determine whether the prediction matches.

In [9]:
results = classifier(sample["review_body"].tolist(), truncation=True)

sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]

sample[["star_rating", "label", "score"]].head(10)

,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


In [10]:
import time

start = time.time()

results = classifier(sample["review_body"].tolist(), truncation=True)

elapsed = time.time() - start

print(f"Time for 500 reviews: {elapsed:.2f} seconds")
print(f"Estimated time for 145,000 reviews: {elapsed * 290 / 60:.2f} minutes")

Time for 500 reviews: 2.94 seconds
Estimated time for 145,000 reviews: 14.20 minutes


It took approximately 3.04 seconds to classify all 500 reviews

At the same rate, processing the full dataset of 145,000 reviews would take approximately 14 minutes

`truncation=True` prevents errors when reviews are too long for the model to process. Since some reviews contain over 5,000 words, it automatically shortens them to fit the model's maximum input length, however, this also means the model might miss important information toward the end of longer reviews.

In [11]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"

sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)

graded = sample[sample["truth"] != "NEUTRAL"]

print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


We exclude 3 star reviews because they often represent neutral or mixed opinions, but the model can only predict POSITIVE or NEGATIVE. So including them would force us to treat a neutral rating as positive or negative, which wouldn't provide a fair evaluation of the model.

The model achieved an accuracy of 87.75%, meaning it correctly classified that percentage of the 400 reviews. While this suggests the model can identify general sentiment, I wouldn't rely on it alone in a real product. Even a relatively accurate model could misclassify important customer complaints, so I would want to examine the types of mistakes it makes before using it for business decisions.

In [12]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


The model made the most mistakes on 4-star reviews, incorrectly classifying 23 out of 100 as NEGATIVE

It also struggled with 5-star reviews, misclassifying 17 as NEGATIVE

This suggests that the model has more difficulty recognizing positive sentiment, especially when customers include criticism in otherwise positive reviews


The model classified 66 of the 3-star reviews as NEGATIVE and 34 as POSITIVE, even though 3 stars generally represents neutral or mixed sentiment

This shows a limitation of binary sentiment classification because the model is forced to choose between positive and negative, even when a customer's opinion falls somewhere in between

In [13]:
mistakes = graded[graded["label"] != graded["truth"]]

for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



Review 1 - The model probably focused on the customer's praise for the game's concept and graphics while overlooking their main complaint about losing progress when the servers stop responding. I think this was the model's mistake because the negative experience clearly explains the 1-star rating.

Review 2 - The review simply says "Enjoys game," which sounds entirely positive despite the 1-star rating. In this case, I think the customer's rating was misleading rather than the model being wrong, since there was no negative language to suggest dissatisfaction.

Review 3 - The customer mentions wasting $7 and being unable to actually play the game, but also says they're sure it's a good game. The model may have focused on that positive statement instead of the overall frustration. I think the model made the mistake here because the customer's negative experience justifies the low rating.

Review 4 -  The customer complains that the game doesn't work properly on their PS Vita but also says it's fun and that they love the series. The model likely focused on the positive language rather than the compatibility issue. I think the model missed the main reason for the negative rating, although the review itself contains mixed opinions.

Review 5 - The customer describes the game's graphics and gameplay before explaining that it feels outdated and didn't entertain their children. The model may have interpreted the neutral descriptions as positive while missing the overall disappointment. I think this was mostly the model's mistake because the review supports the customer's low rating.

In [14]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

results = star_model(sample["review_body"].tolist(), truncation=True)

sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())

off_by = (sample["pred_stars"] - sample["star_rating"]).abs()

print("Within one star:", (off_by <= 1).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574
Within one star: 0.912


The model predicted the exact star rating 57.4% of the time, but it was within one star of the actual rating 91.2% of the time.

I would choose the five-star model because it provides more detailed information about customer satisfaction than simply classifying reviews as positive or negative. Although its exact accuracy is only 57.4%, it predicts within one star 91.2% of the time, making it useful for identifying general trends. This could help a game publisher distinguish between mildly dissatisfied customers and those experiencing serious problems, rather than grouping them all into one negative category.

In [15]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]

results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)

complaints["topic"] = [r["labels"][0] for r in results]

complaints["topic"].value_counts()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

The most common complaints were download or installation issues and bugs or crashes, with 5 reviews each out of the 20 analyzed

Account or login problems followed with 4 reviews, while price and gameplay each had 3. This suggests that technical issues are a major source of customer dissatisfaction, especially problems that prevent people from accessing or properly using their games.



In [16]:
for review in complaints.head(3).itertuples():
    print("Topic:", review.topic)
    print("Review:", review.review_body)
    print()

Topic: price
Review: Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not).

Topic: download or install
Review: I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is.

Topic: gameplay
Review: If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\".



Review 1 - The model's choice makes sense because the customer complains about paying an extra $30 without seeing much improvement in the game, however, they also criticize the interface, graphics, and storyline, so the review could fit under gameplay as well.

Review 2 - I agree with the model's classification because the customer seems to be having trouble running the game despite their computer meeting the system requirements. However, it's unclear whether the problem is specifically with installation or a technical bug.

Review 3 - The model's choice makes sense because the customer's main complaint is that the game is too slow-paced and boring. Unlike the other reviews, this one focuses on the actual gaming experience rather than technical issues or pricing.

I think the model classified these three reviews reasonably well, although the first two show how some complaints can fit into multiple categories.

In [17]:
more_labels = labels + ["technical problems", "HR policy"]

new_results = zsc(
    complaints["review_body"].head(5).tolist(),
    candidate_labels=more_labels
)

for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



After adding the two new labels, the confidence scores decreased for four of the five reviews, even though their top categories stayed the same. For example, "price" dropped from 0.66 to 0.59, and "download or install" dropped from 0.79 to 0.45. The fourth review was the exception, changing from "bugs or crashes" (0.53) to "technical problems" (0.70). This shows how adding an overlapping category can change both the model's predictions and confidence scores.

Since the scores must add up to 1, adding more labels means the model has to redistribute its confidence across more options. This explains why most of the original categories received lower scores, even though the reviews themselves didn't change. It also explains why the new "technical problems" label could become the top prediction when it closely matched an existing category.

This shows that choosing labels carefully is important because they can directly influence the model's predictions. Overlapping categories like "bugs or crashes" and "technical problems" can compete with each other, making the results less consistent. In a real product, I would use clear, distinct categories and test different labels to make sure the results accurately reflect what customers are complaining about.